# Outputs 결과 확인

In [636]:
import re
import json
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(".")

## 1. 전체 결과 로드

In [637]:
score_files = sorted(OUTPUT_DIR.rglob("score.tsv"))
all_dfs = {}
for f in score_files:
    key = str(f.parent.relative_to(OUTPUT_DIR))
    df = pd.read_csv(f, sep='\t')
    all_dfs[key] = df
    print(f"{key}: {len(df)} samples, score={df['score'].mean()*100:.2f}%")

Qwen3-VL-2B-Instruct/MLVU/tp14000_mf128: 2174 samples, score=67.76%
Qwen3-VL-2B-Instruct/MLVU/tp1750_mf16: 2174 samples, score=55.70%
Qwen3-VL-2B-Instruct/MLVU/tp28000_mf256: 2174 samples, score=69.50%
Qwen3-VL-2B-Instruct/MLVU/tp3500_mf32: 2174 samples, score=61.04%
Qwen3-VL-2B-Instruct/MLVU/tp7000_mf64: 2174 samples, score=63.80%
Qwen3-VL-2B-Instruct/Video-MME/tp14000_mf128: 2700 samples, score=61.70%
Qwen3-VL-2B-Instruct/Video-MME/tp1750_mf16: 2700 samples, score=53.81%
Qwen3-VL-2B-Instruct/Video-MME/tp3500_mf32: 2700 samples, score=57.00%
Qwen3-VL-2B-Instruct/Video-MME/tp7000_mf64: 2700 samples, score=59.19%
Qwen3-VL-4B-Instruct/MLVU/tp14000_mf128: 2174 samples, score=70.10%
Qwen3-VL-4B-Instruct/MLVU/tp1750_mf16: 2174 samples, score=58.05%
Qwen3-VL-4B-Instruct/MLVU/tp3500_mf32: 2174 samples, score=62.33%
Qwen3-VL-4B-Instruct/MLVU/tp7000_mf64: 2174 samples, score=67.07%
Qwen3-VL-4B-Instruct/Video-MME/tp14000_mf128: 2700 samples, score=66.11%
Qwen3-VL-4B-Instruct/Video-MME/tp1750_mf1

## 2. 정확도 요약

In [638]:
summary_rows = []
for key, df in all_dfs.items():
    correct = df["score"].sum()
    total = len(df)
    acc = correct / total * 100
    summary_rows.append({"experiment": key, "correct": int(correct), "total": total, "accuracy": f"{acc:.2f}%"})

summary_df = pd.DataFrame(summary_rows)
summary_df

,experiment,correct,total,accuracy
0,Qwen3-VL-2B-Instruct/MLVU/tp14000_mf128,1473,2174,67.76%
1,Qwen3-VL-2B-Instruct/MLVU/tp1750_mf16,1211,2174,55.70%
2,Qwen3-VL-2B-Instruct/MLVU/tp28000_mf256,1511,2174,69.50%
3,Qwen3-VL-2B-Instruct/MLVU/tp3500_mf32,1327,2174,61.04%
4,Qwen3-VL-2B-Instruct/MLVU/tp7000_mf64,1387,2174,63.80%
5,Qwen3-VL-2B-Instruct/Video-MME/tp14000_mf128,1666,2700,61.70%
6,Qwen3-VL-2B-Instruct/Video-MME/tp1750_mf16,1453,2700,53.81%
7,Qwen3-VL-2B-Instruct/Video-MME/tp3500_mf32,1539,2700,57.00%
8,Qwen3-VL-2B-Instruct/Video-MME/tp7000_mf64,1598,2700,59.19%
9,Qwen3-VL-4B-Instruct/MLVU/tp14000_mf128,1524,2174,70.10%


## 3. 결과 요약 테이블 (mf를 column으로)

In [639]:
rows = []
for key, df in all_dfs.items():
    parts = Path(key).parts
    if len(parts) != 3:
        continue
    model, dataset, config = parts
    m = re.search(r'tp(\d+)_mf(\d+)', config)
    if not m:
        continue
    mf = int(m.group(2))
    acc = df["score"].mean() * 100

    rows.append({
        "model": model,
        "dataset": dataset,
        "mf": mf,
        "accuracy": round(acc, 2),
        "total": len(df),
    })

result_df = pd.DataFrame(rows)

pivot = result_df.pivot_table(
    index=["model", "dataset"],
    columns="mf",
    values="accuracy",
)
pivot.columns = [f"mf={c}" for c in pivot.columns]
pivot = pivot.reset_index()

acc_cols = sorted([c for c in pivot.columns if c.startswith("mf=")], key=lambda x: int(x.split("=")[1]))
pivot = pivot[["model", "dataset"] + acc_cols]
pivot

,model,dataset,mf=16,mf=32,mf=64,mf=128,mf=256
0,Qwen3-VL-2B-Instruct,MLVU,55.70,61.04,63.80,67.76,69.50
1,Qwen3-VL-2B-Instruct,Video-MME,53.81,57.00,59.19,61.70,NaN
2,Qwen3-VL-4B-Instruct,MLVU,58.05,62.33,67.07,70.10,NaN
3,Qwen3-VL-4B-Instruct,Video-MME,56.33,60.59,63.07,66.11,67.15


## 4. 에러/OOM으로 인한 누락 샘플 분석

In [640]:
EXPECTED = {"Video-MME": 2700, "MLVU": 2174}

pred_files = sorted(OUTPUT_DIR.rglob("predictions.jsonl"))

error_rows = []
for f in pred_files:
    rel = f.parent.relative_to(OUTPUT_DIR)
    parts = rel.parts
    if len(parts) != 3:
        continue
    model, dataset, config = parts
    m = re.search(r'tp(\d+)_mf(\d+)', config)
    if not m:
        continue
    tp, mf = int(m.group(1)), int(m.group(2))
    exp = EXPECTED.get(dataset)
    if exp is None:
        continue

    lines = [l for l in f.read_text().strip().split('\n') if l.strip()]
    actual = len(lines)
    missing = exp - actual

    if missing == 0:
        status = "Complete"
    elif actual <= 10:
        status = "Failed (early crash, likely OOM)"
    else:
        status = "Partial (interrupted mid-run)"

    error_rows.append({
        "model": model,
        "dataset": dataset,
        "mf": mf,
        "tp": tp,
        "completed": actual,
        "expected": exp,
        "missing": missing,
        "missing_pct": f"{missing / exp * 100:.1f}%",
        "status": status,
    })

error_df = pd.DataFrame(error_rows)

incomplete = error_df[error_df["missing"] > 0].sort_values(["model", "dataset", "mf"])
print(f"=== 총 {len(pred_files)}개 실험 중 {len(incomplete)}개가 불완전 ===")
incomplete[["model", "dataset", "mf", "tp", "completed", "expected", "missing", "missing_pct", "status"]]

=== 총 21개 실험 중 3개가 불완전 ===


,model,dataset,mf,tp,completed,expected,missing,missing_pct,status
4,Qwen3-VL-2B-Instruct,MLVU,512,56000,251,2174,1923,88.5%,Partial (interrupted mid-run)
8,Qwen3-VL-2B-Instruct,Video-MME,256,28000,96,2700,2604,96.4%,Partial (interrupted mid-run)
13,Qwen3-VL-4B-Instruct,MLVU,256,28000,1712,2174,462,21.3%,Partial (interrupted mid-run)


In [641]:
if not error_df.empty:
    status_pivot = error_df.pivot_table(
        index=["model", "dataset"],
        columns="mf",
        values="status",
        aggfunc="first",
    )
    status_pivot.columns = [f"mf={c}" for c in status_pivot.columns]
    status_pivot = status_pivot.reset_index()
    mf_cols = sorted([c for c in status_pivot.columns if c.startswith("mf=")], key=lambda x: int(x.split("=")[1]))
    status_pivot = status_pivot[["model", "dataset"] + mf_cols]
    print("=== 실험별 완료 상태 ===")
    display(status_pivot)
else:
    print("No experiments found.")

=== 실험별 완료 상태 ===


,model,dataset,mf=16,mf=32,mf=64,mf=128,mf=256,mf=512
0,Qwen3-VL-2B-Instruct,MLVU,Complete,Complete,Complete,Complete,Complete,Partial (interrupted mid-run)
1,Qwen3-VL-2B-Instruct,Video-MME,Complete,Complete,Complete,Complete,Partial (interrupted mid-run),NaN
2,Qwen3-VL-4B-Instruct,MLVU,Complete,Complete,Complete,Complete,Partial (interrupted mid-run),NaN
3,Qwen3-VL-4B-Instruct,Video-MME,Complete,Complete,Complete,Complete,Complete,NaN
